# Lesson 5 — The zero-probability problem

MLE gives exactly 0 to any word pair it never saw. A sentence's probability is a *product* of word steps, so one
zero makes the whole sentence "impossible" and its perplexity infinite. Smoothing keeps a little probability back
for things the model hasn't seen yet.

In [1]:
import re, math, random
from collections import Counter, defaultdict

In [2]:
def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

In [3]:
def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

The vocabulary `V` is every training word plus `</s>`. Smoothing spreads probability over all of it.

In [4]:
sents = to_sentences(open("corpus.txt").read())
foll = build(sents, n=2)
V = set(t for s in sents for t in to_tokens(s)) | {"</s>"}     # the vocabulary

## MLE vs add-k

- **MLE:** count / total, zero when the pair was never seen.
- **Add-k (Laplace when k=1):** pretend every word in V followed the context k extra times:
  (count + k) / (total + k·|V|). Nothing is ever zero, and seen words still rank higher.

In [5]:
def prob_mle(ctx, w):
    c = foll[(ctx,)]; total = sum(c.values())
    return c[w] / total if total else 0.0

def prob_addk(ctx, w, k=1.0):                                  # add-k / Laplace smoothing
    c = foll[(ctx,)]; total = sum(c.values())
    return (c[w] + k) / (total + k * len(V))

The test case: "drinks coffee" never appears. The corpus only has "drinks his coffee" and "drinks green tea".

In [6]:
# 'drinks coffee' never occurs (the corpus has 'drinks his coffee' and 'drinks green tea')
print("MLE   P(coffee | drinks) =", prob_mle("drinks", "coffee"), " <- zero, so 'she drinks coffee' is 'impossible'")
print("add-1 P(coffee | drinks) =", round(prob_addk("drinks", "coffee"), 5), " <- small but non-zero")
print("add-1 P(green  | drinks) =", round(prob_addk("drinks", "green"), 5), " <- a word we DID see still ranks higher")

MLE   P(coffee | drinks) = 0.0  <- zero, so 'she drinks coffee' is 'impossible'
add-1 P(coffee | drinks) = 0.00581  <- small but non-zero
add-1 P(green  | drinks) = 0.01163  <- a word we DID see still ranks higher


Here |V| = 170 and "drinks" was seen twice, so add-1 gives (0+1)/(2+170) = 0.00581 for the unseen "coffee" and
(1+1)/172 = 0.01163 for the seen "green". The catch is that add-1 shifts a lot of mass away from what we actually
saw (here the two observed words keep only 4/172 between them, and 168/172 goes to unseen words!). Real systems use a small k, or back off to a shorter context
when the longer one is unseen.

## Test yourself

**1. Why is one zero probability so damaging?**
Sentence probability is a product of word steps, so a single 0 makes the whole sentence 0. Its log is −∞, and
perplexity becomes infinite.

**2. Write the add-k formula and say what each part does.**
P = (count(ctx, w) + k) / (count(ctx) + k·|V|). The +k in the numerator gives every word some mass. The k·|V| in
the denominator keeps the probabilities summing to 1.

**3. What is the downside of add-1 on a big vocabulary?**
It takes too much probability from the words you really observed and spreads it thinly over thousands of unseen
ones. That hurts the common cases. A smaller k (like 0.5 or 0.01), or back-off/interpolation, works better.

## My notes

The key idea: smoothing just adds k fake counts to every word, so nothing is impossible. What surprised me is how
heavy-handed add-1 is with a tiny context. "drinks" has only 2 real observations against 170 fake ones.